# OpenGeoSys in 10 minutes — the shorter excavation demo

**One Jupyter notebook. Four steps. Preprocessing → Processing → Postprocessing.**

This is the same **65 m × 30 m excavation example** used in the original course, but with the meshes and boundary extraction already prepared. We focus on four actions: load a mesh, inspect the project, run OGS, and plot the result.

**TEC environment:** Python **3.10.12** and `ogstools==0.7.1` (the newest OGSTools 0.8.x requires Python 3.11+). `ogs` must be available on PATH.

Open this notebook inside the course folder. If `import ogstools` fails, see `OGSTOOLS_QUICK_SETUP.md`.

**For the audience:** Each cell has one visible task. The advanced nine-step training stays available separately.

## 1 — Preprocessing: inspect the finite-element mesh

The Gmsh mesh has **already been converted** into the OGS bulk mesh. We now load it using OGSTools. In a normal project, geometry generation and conversion are upstream preprocessing tasks.

**Look for:** the thin impermeable wall and the connected domain underneath its tip.

In [ ]:
from pathlib import Path
import ogstools as ot

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "models" / "excavation.prj").exists()), None)
assert ROOT is not None, "Open the training folder in VS Code."
bulk = ot.Mesh.read(ROOT / "mesh" / "excavation.vtu")
print(f"Bulk mesh: {bulk.n_points} nodes, {bulk.n_cells} elements")
ot.plot.contourf(bulk, ot.variables.material_id, show_edges=True)

## 2 — Model setup: read the OGS project

The project file contains the diffusion process, material properties, solver and two upper boundary conditions.

**Reference case:** potential **60** at the top-left, **30** at the top-right. The wall is impermeable.

Here we reuse the prepared project rather than retyping many XML settings.

In [ ]:
RESULTS = ROOT / "quick_demo_results"
RESULTS.mkdir(exist_ok=True)
project = ot.Project(
    input_file=ROOT / "models" / "excavation.prj",
    output_file=ROOT / "models" / "quick_demo.prj",
    output_dir=RESULTS,
)
project.write_input()
print("OGS project prepared:", ROOT / "models" / "quick_demo.prj")
print("BC values: left = 60, right = 30 (model potential units)")

## 3 — Processing: run the solver

One command starts OpenGeoSys through OGSTools.

**Look for:** a successful OGS process and a new VTU result under `quick_demo_results/`.

In [ ]:
project.run_model(args=f"-o {RESULTS}")
result_files = sorted(RESULTS.glob("excavation_diffusion_ts_1_*.vtu"))
assert result_files, "OGS did not produce the expected final VTU."
print("OGS status:", project.status)
print("Result:", result_files[-1].name)

## 4 — Postprocessing: plot the computed potential

OGSTools loads the **actual simulated VTU** and plots the potential field.

**Discussion:** How does the impermeable wall influence the gradients? Where is the potential highest?

In [ ]:
result = ot.Mesh.read(result_files[-1])
ot.plot.contourf(result, "pressure")
print("PASS: Preprocessing → OGS simulation → computed field")

### Finished — what did we learn?

1. **Preprocessing:** load and inspect an OGS mesh.
2. **Model setup:** reuse a documented OGS project and boundary conditions.
3. **Processing:** execute OGS in Python using `ot.Project.run_model()`.
4. **Postprocessing:** display the simulated field using `ot.plot.contourf()`.

The output variable `pressure` is the model's abstract potential; this example is not calibrated to kPa or m/s.

**Live-demo rule:** If a step fails, stop and show the last valid result. Do not spend the workshop debugging the environment.

**References:** [OGSTools 0.7.1 release notes](https://ogstools.opengeosys.org/stable/releases/ogstools-0.7.1.html) · [Running OGS with OGSTools](https://ogstools.opengeosys.org/0.7.0/auto_user-guide/plot_simulate.html)